# Notebook 1: Setup & Data (Kaggle + Google Cloud Storage)

This notebook:
1. Installs dependencies
2. **Auto-downloads Sen1Floods11** from `gs://sen1floods11` (no manual download needed)
3. Sets up paths for Kaggle
4. Verifies the data by visualizing a sample

---

**Dataset**: Sen1Floods11 — hosted on `gs://sen1floods11` (public GCS bucket)

## Step 1: Environment Setup

In [1]:
import sys, os, subprocess

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/flood_detection'
elif IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/flood_detection'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '.'))

os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'Platform     : {"Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "Local"}')
print(f'Project root : {PROJECT_ROOT}')

Platform     : Kaggle
Project root : /kaggle/working/flood_detection


## Step 2: Install Dependencies

In [2]:
!pip install -q timm albumentations rasterio rioxarray earthengine-api geedim geemap einops tqdm joblib
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.4 MB/s eta 0:00:00a 0:00:01
Dependencies installed.


## Step 3: Download Sen1Floods11 from Google Cloud Storage

Downloads the **hand-labeled subset** directly from `gs://sen1floods11` (public bucket).  
Only downloads what's needed for training — skips files already present.

| Folder | Size | Purpose |
|--------|------|---------|
| S1Hand | ~1.8 GB | Sentinel-1 SAR images (model input) |
| LabelHand | ~0.5 GB | Flood labels (ground truth) |
| JRCWaterHand | ~0.5 GB | Permanent water data |
| S2Hand | ~3.5 GB | Sentinel-2 (for MNDWI auxiliary task) |
| splits | tiny | Train/val/test split CSV files |

**Total: ~6 GB** — downloads in ~10-15 min on Kaggle.

---
# Downloading Sen1Floods11 Data Only Once 
---

In [3]:
# ============================================================
# Auto-download Sen1Floods11 from GCS (public bucket, no auth needed)
# ============================================================

GCS_BUCKET  = 'gs://sen1floods11/v1.1'
LOCAL_DATA  = '/kaggle/working/sen1floods11'

# What to download: (GCS subdirectory, local subdirectory)
DOWNLOADS = [
    ('data/flood_events/HandLabeled/S1Hand',       'HandLabeled/S1Hand'),
    ('data/flood_events/HandLabeled/LabelHand',    'HandLabeled/LabelHand'),
    ('data/flood_events/HandLabeled/JRCWaterHand', 'HandLabeled/JRCWaterHand'),
    ('data/flood_events/HandLabeled/S2Hand',       'HandLabeled/S2Hand'),
    ('splits/flood_handlabeled',                   'splits'),
]

def download_from_gcs(gcs_subdir, local_dir):
    """Download a folder from GCS using gsutil. Skips if already present."""
    if os.path.isdir(local_dir) and len(os.listdir(local_dir)) > 5:
        count = len(os.listdir(local_dir))
        print(f'  SKIP  {os.path.basename(local_dir):20s} — already cached ({count} files)')
        return
    os.makedirs(local_dir, exist_ok=True)
    gcs_path = f'{GCS_BUCKET}/{gcs_subdir}'
    print(f'  DOWNLOADING {os.path.basename(local_dir)} from {gcs_path} ...')
    result = subprocess.run(
        ['gsutil', '-m', '-q', 'cp', '-r', f'{gcs_path}/*', f'{local_dir}/'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # Fallback: try without wildcard (some gsutil versions handle dirs differently)
        subprocess.run(
            ['gsutil', '-m', '-q', 'cp', '-r', gcs_path, os.path.dirname(local_dir) + '/'],
            capture_output=True, text=True
        )
    count = len(os.listdir(local_dir))
    print(f'  DONE  {os.path.basename(local_dir):20s} — {count} files')

print('Downloading Sen1Floods11 from Google Cloud Storage...\n')
for gcs_sub, local_sub in DOWNLOADS:
    download_from_gcs(gcs_sub, os.path.join(LOCAL_DATA, local_sub))

print('\nDownload complete!')


  DOWNLOADING S1Hand from gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand ...
  DONE  S1Hand               — 446 files
  DOWNLOADING LabelHand from gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand ...
  DONE  LabelHand            — 446 files
  DOWNLOADING JRCWaterHand from gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand ...
  DONE  JRCWaterHand         — 446 files
  DOWNLOADING S2Hand from gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand ...
  DONE  S2Hand               — 446 files
  DOWNLOADING splits from gs://sen1floods11/v1.1/splits/flood_handlabeled ...
  DONE  splits               — 4 files

Download complete!


---
# GEE Setup

In [6]:
### Initiate GEE

import ee
import json

# Load credentials
SERVICE_ACCOUNT_FILE = '/kaggle/input/datasets/yash10chawla/last-try-key/gee-kaggle-project-fbe0b036073d.json'

credentials = ee.ServiceAccountCredentials(
    email=None,  # can be None for JSON-based auth
    key_file=SERVICE_ACCOUNT_FILE
)

ee.Initialize(credentials)

In [7]:
### Checks if GEE is working 

print(ee.Image("USGS/SRTMGL1_003").getInfo())

{'type': 'Image', 'bands': [{'id': 'elevation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [1296001, 417601], 'crs': 'EPSG:4326', 'crs_transform': [0.0002777777777777778, 0, -180.0001388888889, 0, -0.0002777777777777778, 60.00013888888889]}], 'version': 1641990767055141, 'id': 'USGS/SRTMGL1_003', 'properties': {'system:visualization_0_min': '0.0', 'type_name': 'Image', 'keywords': ['dem', 'elevation', 'geophysical', 'nasa', 'srtm', 'topography', 'usgs'], 'thumb': 'https://mw1.google.com/ges/dd/images/SRTM90_V4_thumb.png', 'description': '<p>The Shuttle Radar Topography Mission (SRTM, see <a href="https://onlinelibrary.wiley.com/doi/10.1029/2005RG000183/full">Farr\net al. 2007</a>)\ndigital elevation data is an international research effort that\nobtained digital elevation models on a near-global scale. This\nSRTM V3 product (SRTM Plus) is provided by NASA JPL\nat a resolution of 1 arc-second (approximately 30m).</p><p>This dataset

---
Calude Code to download DEM,Slope, HAND


In [8]:
SRC_DIR   = '/kaggle/input/datasets/yash10chawla/flood-detection-src'
DATA_ROOT = '/kaggle/working/sen1floods11'

In [9]:
!python {SRC_DIR}/fetch_aux_bands.py --data_root {DATA_ROOT}

[fetch_aux] 446 chips to consider | bands=['DEM', 'Slope', 'HAND']
[GEE] Earth Engine initialized (service account).
/usr/local/lib/python3.12/dist-packages/geedim/mask.py:579: FutureWarning: 'MaskedImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  super().__init__(ee_image)
NASA/NASADEM_HGT/001: 100%|█████████████████████████████|1/1 tiles [00:00<00:00]
  [   1/446] Bolivia_103757 :: DEM ✓
/usr/local/lib/python3.12/dist-packages/geedim/mask.py:579: FutureWarning: 'MaskedImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  super().__init__(ee_image)
100%|███████████████████████████████████████████████████|1/1 tiles [00:00<00:00]
/usr/local/lib/python3.12/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)
  [   1/446] Bolivia_103757 :: Slope ✓
/usr/local/lib/python3.12/dist-packages/geedim/mask.p

---
New Permanent dataset for Kaggle 


In [10]:
# ============================================================
# ONE-TIME: Save downloaded data as permanent Kaggle Dataset
# Run this cell ONCE, then delete it from your notebook
# ============================================================
import json, subprocess

metadata = {
    "title": "Final-SenData",
    "id": "yash10chawla/final-sendata",
    "licenses": [{"name": "CC0-1.0"}]
}
with open('/kaggle/working/sen1floods11/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Creating Kaggle dataset 'Final-SenData'... (this may take 5-10 min)")
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', '/kaggle/working/sen1floods11', '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "Upload triggered.")
if result.stderr:
    print("STDERR:", result.stderr)
print("\nDone! Go to kaggle.com → Datasets to confirm it was created.")
print("Then add 'final-sendata' dataset to this notebook via: Notebook Settings → Add Data")

Creating Kaggle dataset 'Final-SenData'... (this may take 5-10 min)
Starting upload for file splits.zip
Upload successful: splits.zip (4KB)
Starting upload for file HandLabeled.zip
Upload successful: HandLabeled.zip (2GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/yash10chawla/final-sendata

STDERR: 
  0%|          | 0.00/4.30k [00:00<?, ?B/s]
100%|██████████| 4.30k/4.30k [00:00<00:00, 19.9kB/s]

  0%|          | 0.00/1.68G [00:00<?, ?B/s]
  1%|▏         | 23.1M/1.68G [00:00<00:07, 238MB/s]
  3%|▎         | 45.8M/1.68G [00:00<00:10, 173MB/s]
  4%|▎         | 63.3M/1.68G [00:00<00:11, 157MB/s]
  5%|▍         | 78.8M/1.68G [00:00<00:11, 147MB/s]
  5%|▌         | 93.0M/1.68G [00:00<00:12, 135MB/s]
  6%|▌         | 106M/1.68G [00:00<00:12, 130MB/s] 
  7%|▋         | 121M/1.68G [00:00<00:12, 137MB/s]
  8%|▊         | 135M/1.68G [00:00<00:11, 139MB/s]
  9%|▊         | 148M/1.68G [00:01<00:12, 135MB/s]
  9%|▉         | 163M/1.68G [00:01<00:

---
# Calling the data 
---

In [3]:
# ============================================================
# Load Sen1Floods11 from permanent Kaggle dataset (BTP-SenData)
# No downloading needed — data lives in /kaggle/input/btp-sendata
# ============================================================
LOCAL_DATA = '/kaggle/input/datasets/yash10chawla/final-sendata'

print('Using Sen1Floods11 from Kaggle dataset: Final-SenData\n')
all_ok = True
checks = [
    ('HandLabeled/S1Hand',       446),
    ('HandLabeled/LabelHand',    446),
    ('HandLabeled/JRCWaterHand', 446),
    ('HandLabeled/S2Hand',       446),
    ('HandLabeled/DEM',          446),
    ('HandLabeled/HAND',         446),
    ('HandLabeled/Slope',        446),
    ('splits',                     4),
]
for sub, expected in checks:
    path = os.path.join(LOCAL_DATA, sub)
    if os.path.isdir(path):
        count = len(os.listdir(path))
        status = '✓' if count >= expected else '⚠'
        print(f'  {status} {sub:35s} — {count} files')
    else:
        print(f'  ✗ {sub:35s} — NOT FOUND at {path}')
        all_ok = False

print(f'\n{"Data ready!" if all_ok else "ERROR: Some folders missing — check dataset path."}')

Using Sen1Floods11 from Kaggle dataset: Final-SenData

  ✓ HandLabeled/S1Hand                  — 446 files
  ✓ HandLabeled/LabelHand               — 446 files
  ✓ HandLabeled/JRCWaterHand            — 446 files
  ✓ HandLabeled/S2Hand                  — 446 files
  ✓ HandLabeled/DEM                     — 446 files
  ✓ HandLabeled/HAND                    — 446 files
  ✓ HandLabeled/Slope                   — 446 files
  ✓ splits                              — 4 files

Data ready!


## Step 4: Set Paths

All paths auto-configured for Kaggle. No changes needed if you ran Step 3.

In [4]:
# ---- PATHS (auto-configured for Kaggle + GCS download) ----
# DATA_ROOT  → contains HandLabeled/ subdirectory
# SPLITS_ROOT → contains flood_train_data.csv, flood_valid_data.csv, flood_test_data.csv

DATA_ROOT   = LOCAL_DATA                                   # /kaggle/working/sen1floods11
SPLITS_ROOT = os.path.join(LOCAL_DATA, 'splits')           # .../splits/

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
OUTPUT_DIR     = os.path.join(PROJECT_ROOT, 'outputs')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'DATA_ROOT      : {DATA_ROOT}')
print(f'SPLITS_ROOT    : {SPLITS_ROOT}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')
print(f'OUTPUT_DIR     : {OUTPUT_DIR}')
print()

# Verify downloaded data
hand_dir = os.path.join(DATA_ROOT, 'HandLabeled')
if os.path.isdir(hand_dir):
    print('Downloaded data:')
    for sub in ['S1Hand', 'LabelHand', 'S2Hand', 'JRCWaterHand', 'DEM', 'Slope','HAND']:
        sub_path = os.path.join(hand_dir, sub)
        if os.path.isdir(sub_path):
            count = len([f for f in os.listdir(sub_path) if f.endswith('.tif')])
            print(f'  {sub:20s}: {count} .tif files')
        else:
            print(f'  {sub:20s}: NOT FOUND')
else:
    print(f'ERROR: HandLabeled dir not found at {hand_dir}')
    print('Re-run Step 3 (GCS download) above.')

DATA_ROOT      : /kaggle/input/datasets/yash10chawla/final-sendata
SPLITS_ROOT    : /kaggle/input/datasets/yash10chawla/final-sendata/splits
CHECKPOINT_DIR : /kaggle/working/flood_detection/checkpoints
OUTPUT_DIR     : /kaggle/working/flood_detection/outputs

Downloaded data:
  S1Hand              : 446 .tif files
  LabelHand           : 446 .tif files
  S2Hand              : 446 .tif files
  JRCWaterHand        : 446 .tif files
  DEM                 : 446 .tif files
  Slope               : 446 .tif files
  HAND                : 446 .tif files


## Step 5: Add src/ to Python Path

Upload the `src/` folder to Kaggle as a **Dataset** or as a **Utility Script**, then set the path below.

In [5]:
# Add src/ to Python path so we can import project modules
# On Kaggle: upload src/ as a Kaggle Dataset, then set path below

if IN_KAGGLE:
    # Option 1: If you uploaded src/ as a Kaggle Dataset named "flood-detection-src"
    SRC_PATH = '/kaggle/input/datasets/yash10chawla/flood-detection-src'
    # Option 2: If src/ is in the notebook's working directory
    if not os.path.isdir(SRC_PATH):
        SRC_PATH = '/kaggle/working/src'
    # Option 3: If running from the repo directly
    if not os.path.isdir(SRC_PATH):
        SRC_PATH = os.path.join(os.path.dirname(os.getcwd()), 'src')
elif IN_COLAB:
    SRC_PATH = f'{PROJECT_ROOT}/src'
else:
    SRC_PATH = os.path.join(os.path.dirname(os.getcwd()), 'src')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'src path: {SRC_PATH}')
if os.path.isdir(SRC_PATH):
    print(f'  files: {sorted(os.listdir(SRC_PATH))}')
else:
    print('  WARNING: src/ directory not found!')
    print('  Upload the src/ folder as a Kaggle Dataset and update SRC_PATH above.')

src path: /kaggle/input/datasets/yash10chawla/flood-detection-src
  files: ['augmentations.py', 'dataset.py', 'fetch_aux_bands.py', 'inference.py', 'losses.py', 'metrics.py', 'model.py', 'train.py']
